
# Cloud Storage (GCS) Hands-On Lab
### GCP Training — Day 1, Module 1

Files that don't belong in a database — product images, invoice PDFs, export CSVs on their way to
BigQuery, model checkpoints, log archives — are objects, not rows. This lab works through the
things you actually need to know to run GCS for real: buckets and objects, storage classes and
lifecycle rules, versioning, access control, signed URLs, and (in the second half) some of the
less-taught stuff — retries, batching, and Transfer Manager for moving a lot of data fast.

Everything goes through `google-cloud-storage`, no CLI. Run top to bottom — the Configure cell
asks for your project ID and a bucket name, nothing to hand-edit.

You'll need `roles/storage.admin` (or better) on the project.


## 1. Auth

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("authenticated")
else:
    print("not in Colab — assuming ADC is set up (gcloud auth application-default login)")

!gcloud config list


## 2. Install & import

In [ ]:
!pip install --quiet "google-cloud-storage>=2.10.0"

In [ ]:

from google.cloud import storage
from google.cloud.storage import transfer_manager
from google.api_core import exceptions as gcs_exceptions
from google.api_core.retry import Retry
from datetime import timedelta
import time, os



## 3. Project & bucket
Bucket names are global across all of GCS, so we tack a timestamp on the default to dodge collisions.


In [ ]:

PROJECT_ID = input("GCP Project ID: ").strip()
REGION = input("Region [asia-south1]: ").strip() or "asia-south1"

_default_bucket = f"gcs-training-demo-{int(time.time())}"
BUCKET_NAME = input(f"Bucket name [{_default_bucket}]: ").strip() or _default_bucket

!gcloud config set project {PROJECT_ID}
client = storage.Client(project=PROJECT_ID)
print("client project:", client.project, "| target bucket:", BUCKET_NAME)


## 4. Buckets

In [ ]:

bucket = client.bucket(BUCKET_NAME)
bucket.storage_class = "STANDARD"
new_bucket = client.create_bucket(bucket, location=REGION)
print(f"created gs://{new_bucket.name} in {new_bucket.location}")



> 🖥️ **Console:** Cloud Storage > Buckets — new bucket should be listed with matching location/class.


In [ ]:

# list, get metadata, tweak config — three quick ones back to back
for b in client.list_buckets():
    print(" -", b.name)

bucket = client.get_bucket(BUCKET_NAME)
print("\nlocation:", bucket.location, "| class:", bucket.storage_class, "| created:", bucket.time_created)

bucket.labels = {"env": "training", "module": "gcs-lab"}
bucket.iam_configuration.uniform_bucket_level_access_enabled = True
bucket.patch()
print("\nlabels + uniform access set:", bucket.labels)



> 🖥️ Bucket's Configuration tab should now show the labels and **Access control: Uniform**.



## 5. Upload / download
Local file, in-memory string, download both ways — the four combinations you'll actually use.


In [ ]:

with open("sample.txt", "w") as f:
    f.write("Hello from the GCS lab!\n")

blob = bucket.blob("uploads/sample.txt")
blob.upload_from_filename("sample.txt")

blob2 = bucket.blob("uploads/inmemory.json")
blob2.upload_from_string('{"lab": "gcs", "day": 1}', content_type="application/json")

print("uploaded:", blob.name, "and", blob2.name)


In [ ]:

blob.download_to_filename("sample_downloaded.txt")
print(open("sample_downloaded.txt").read())

print(bucket.blob("uploads/inmemory.json").download_as_text())



> 🖥️ Objects tab → `uploads/` should show both files; click `inmemory.json` and check
> Content-Type is `application/json` even though we never touched a local file for it.


## 6. Listing

In [ ]:

print("everything:")
for b in bucket.list_blobs():
    print(" ", b.name, b.size, "bytes")

print("\njust uploads/:")
for b in bucket.list_blobs(prefix="uploads/"):
    print(" ", b.name)



There's no such thing as a folder in GCS — `uploads/` is just a naming convention. The console's
folder view is doing exactly this `prefix=` filtering under the hood.


## 7. Copy, rename, delete

In [ ]:

source = bucket.blob("uploads/sample.txt")
copied = bucket.copy_blob(source, bucket, "uploads/sample_copy.txt")
print("copied ->", copied.name)

# no native rename in GCS — this is copy + delete under the hood
renamed = bucket.rename_blob(copied, "archive/sample_renamed.txt")
print("renamed ->", renamed.name)

bucket.blob("archive/sample_renamed.txt").delete()
print("deleted archive/sample_renamed.txt")



> 🖥️ Watch the Objects tab through this one — `sample_copy.txt` appears, then disappears the
> moment it's "renamed" and a new `archive/...` object appears in its place. Deletion is immediate
> and (no versioning yet) unrecoverable.


## 8. Metadata

In [ ]:

blob = bucket.blob("uploads/sample.txt")
blob.reload()
print("size:", blob.size, "| type:", blob.content_type, "| md5:", blob.md5_hash, "| gen:", blob.generation)

blob.metadata = {"lab-owner": "instructor", "purpose": "gcs-demo"}
blob.content_type = "text/plain"
blob.patch()
print("custom metadata:", blob.metadata)



> 🖥️ The object's details panel in the console is just a rendered view of everything `blob.reload()`
> pulled back — custom metadata included, editable right there too.



## 9. Storage classes

| Class | Min. duration | Access pattern | Cost shape |
|---|---|---|---|
| STANDARD | none | hot | highest storage, free retrieval |
| NEARLINE | 30 days | ~monthly | cheaper storage, small retrieval fee |
| COLDLINE | 90 days | ~quarterly | cheaper still, bigger retrieval fee |
| ARCHIVE | 365 days | rare/compliance | cheapest storage, priciest retrieval |

Set per-object or as a bucket default. Changing class is a metadata rewrite, not a data move.


In [ ]:

bucket.storage_class = "NEARLINE"
bucket.patch()
print("bucket default is now", bucket.storage_class, "— only affects NEW objects, not existing ones")

blob.update_storage_class("COLDLINE")
print("uploads/sample.txt is now", bucket.get_blob("uploads/sample.txt").storage_class)



## 10. Lifecycle rules
Move to Coldline after 30 days, delete after 365 — automatic, no cron job required.


In [ ]:

bucket.lifecycle_rules = [
    {"action": {"type": "SetStorageClass", "storageClass": "COLDLINE"}, "condition": {"age": 30}},
    {"action": {"type": "Delete"}, "condition": {"age": 365, "isLive": True}},
]
bucket.patch()

bucket.reload()
for rule in bucket.lifecycle_rules:
    print(rule)



> 🖥️ Buckets > Lifecycle tab renders these two rules in plain English.


In [ ]:

bucket.clear_lifecycle_rules()
bucket.patch()
print("rules now:", bucket.lifecycle_rules)



## 11. Versioning
Once on, every write to the same object name creates a new **generation** instead of overwriting.


In [ ]:

bucket.versioning_enabled = True
bucket.patch()

blob = bucket.blob("uploads/versioned.txt")
blob.upload_from_string("version 1")
blob.upload_from_string("version 2")
blob.upload_from_string("version 3")

for b in client.list_blobs(bucket, prefix="uploads/versioned.txt", versions=True):
    print("gen", b.generation, "-", b.size, "bytes")



> 🖥️ Object's version list in the console shows all 3 generations — same thing, browsable without code.


In [ ]:

versions = list(client.list_blobs(bucket, prefix="uploads/versioned.txt", versions=True))
oldest = sorted(versions, key=lambda b: b.generation)[0]
bucket.blob(oldest.name, generation=oldest.generation).delete()
print("deleted gen", oldest.generation, "— the other two are untouched")



## 12. Signed URLs
Temporary, credential-less access — the mechanism behind "click here to download" links that
expire. Note: signing needs either a service account key file or impersonation; plain end-user
credentials in Colab can't sign directly, which is exactly the wall you'll hit below if you skip
the impersonation setup.


In [ ]:

import google.auth, google.auth.transport.requests, requests
from google.auth import impersonated_credentials

# whoami, reliably — gcloud config get-value account is flaky under Colab's auth flow
credentials, _ = google.auth.default()
credentials.refresh(google.auth.transport.requests.Request())
me = requests.get(
    "https://www.googleapis.com/oauth2/v1/userinfo",
    headers={"Authorization": f"Bearer {credentials.token}"},
).json()["email"]

SIGNING_SA = f"training-url-signer-{_suffix if '_suffix' in dir() else int(time.time())}"
SIGNING_SA_EMAIL = f"{SIGNING_SA}@{PROJECT_ID}.iam.gserviceaccount.com"

!gcloud iam service-accounts create {SIGNING_SA} --display-name="URL signer" --project={PROJECT_ID} --quiet
!gcloud storage buckets add-iam-policy-binding gs://{BUCKET_NAME} --member="serviceAccount:{SIGNING_SA_EMAIL}" --role="roles/storage.objectViewer" --quiet
!gcloud iam service-accounts add-iam-policy-binding {SIGNING_SA_EMAIL} --member="user:{me}" --role="roles/iam.serviceAccountTokenCreator" --project={PROJECT_ID} --quiet

print("waiting for IAM to catch up...")
time.sleep(45)

signing_creds = impersonated_credentials.Credentials(
    source_credentials=credentials,
    target_principal=SIGNING_SA_EMAIL,
    target_scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
print("ready to sign as", SIGNING_SA_EMAIL)


In [ ]:

signed_get = blob.generate_signed_url(version="v4", expiration=timedelta(minutes=15), method="GET", credentials=signing_creds)
print(signed_get)



> 🖥️ Paste that into an incognito window — loads with zero sign-in prompt. The signature *is* the
> credential, good for 15 minutes.


In [ ]:

upload_blob = bucket.blob("uploads/via_signed_url.txt")
signed_put = upload_blob.generate_signed_url(
    version="v4", expiration=timedelta(minutes=15), method="PUT",
    content_type="text/plain", credentials=signing_creds,
)
print(signed_put)
# test with: curl -X PUT --upload-file localfile.txt "SIGNED_URL"



## 13. CORS
Only matters if a browser app on a different origin needs to call GCS directly.


In [ ]:

bucket = client.get_bucket(BUCKET_NAME)
bucket.cors = [{"origin": ["https://example.com"], "method": ["GET", "HEAD"],
                "responseHeader": ["Content-Type"], "maxAgeSeconds": 3600}]
bucket.patch()
print(bucket.cors)



## 14. Compose
Stitch objects together server-side — no download/re-upload round trip. Handy after a chunked
parallel upload (Section 18) leaves you with pieces to join.


In [ ]:

p1 = bucket.blob("compose/part1.txt"); p1.upload_from_string("Hello, ")
p2 = bucket.blob("compose/part2.txt"); p2.upload_from_string("World!")

combined = bucket.blob("compose/combined.txt")
combined.compose([p1, p2])
print(combined.download_as_text())



## Part B — the stuff most tutorials skip

### 15. Batch requests
5 objects, one metadata patch each — that's 5 HTTP round trips unless you batch them.


In [ ]:

for i in range(5):
    bucket.blob(f"batch-demo/file_{i}.txt").upload_from_string(f"content {i}")

blobs = [bucket.blob(f"batch-demo/file_{i}.txt") for i in range(5)]
with client.batch():
    for b in blobs:
        b.metadata = {"batch-tagged": "true"}
        b.patch()

check = bucket.blob("batch-demo/file_0.txt")
check.reload()
print(check.metadata)


In [ ]:

with client.batch():
    for b in blobs:
        b.delete()
print("all 5 gone, one batch request")



Batching covers metadata calls (patch/delete) — content upload/download is a different animal,
covered next.



### 16. Transfer Manager — many files in parallel
Ten small files, sequential vs. parallel, so the speedup is something you actually measure instead
of just believing.


In [ ]:

os.makedirs("transfer_demo", exist_ok=True)
filenames = []
for i in range(10):
    fname = f"transfer_demo/part_{i:02d}.txt"
    with open(fname, "w") as f:
        f.write(f"payload {i}\n" * 1000)
    filenames.append(fname)

start = time.time()
for fname in filenames:
    bucket.blob(f"transfer-demo/{os.path.basename(fname)}").upload_from_filename(fname)
sequential_seconds = time.time() - start
print(f"sequential: {sequential_seconds:.2f}s")

with client.batch():
    for fname in filenames:
        bucket.blob(f"transfer-demo/{os.path.basename(fname)}").delete()


In [ ]:

start = time.time()
results = transfer_manager.upload_many_from_filenames(
    bucket, filenames, source_directory="", blob_name_prefix="transfer-demo/", max_workers=8,
)
parallel_seconds = time.time() - start

failures = [r for r in results if r is not None]
print(f"parallel: {parallel_seconds:.2f}s ({sequential_seconds/parallel_seconds:.1f}x), {len(failures)} failures")



> 🖥️ The console can't tell you *how* those 10 files got uploaded — the timing numbers above are
> the only real evidence Transfer Manager did anything.


In [ ]:

# same idea, download direction
blob_names = [f"transfer-demo/{os.path.basename(f)}" for f in filenames]
os.makedirs("transfer_demo_downloaded", exist_ok=True)

start = time.time()
results = transfer_manager.download_many_to_path(bucket, blob_names, destination_directory="transfer_demo_downloaded/", max_workers=8)
print(f"parallel download: {time.time()-start:.2f}s, {sum(1 for r in results if r is not None)} failures")



### 17. Transfer Manager — one big file, chunked
Different problem: instead of parallelizing *across* files, split *one* large file into chunks and
move them concurrently. Useful for big exports or model checkpoints.


In [ ]:

size_mb = 50
large_file = "transfer_demo/large_file.bin"
with open(large_file, "wb") as f:
    f.write(os.urandom(1024 * 1024) * size_mb)

large_blob_name = "transfer-demo/large_file.bin"
start = time.time()
transfer_manager.upload_chunks_concurrently(large_file, bucket.blob(large_blob_name), chunk_size=8*1024*1024, max_workers=4)
print(f"chunked upload of {size_mb}MB: {time.time()-start:.2f}s")


In [ ]:

downloaded = "transfer_demo_downloaded/large_file_downloaded.bin"
start = time.time()
transfer_manager.download_chunks_concurrently(bucket.blob(large_blob_name), downloaded, chunk_size=8*1024*1024, max_workers=4)
print(f"chunked download: {time.time()-start:.2f}s")

assert os.path.getsize(downloaded) == os.path.getsize(large_file)
print("sizes match")



## Cleanup
Kills every object (and version) in the bucket, then the bucket, then the local scratch files.


In [ ]:

for b in client.list_blobs(bucket, versions=True):
    bucket.blob(b.name, generation=b.generation).delete()
bucket.delete()

import shutil
shutil.rmtree("transfer_demo", ignore_errors=True)
shutil.rmtree("transfer_demo_downloaded", ignore_errors=True)
print(f"{BUCKET_NAME} and everything in it: gone")


In [ ]:

try:
    client.get_bucket(BUCKET_NAME)
    print("still exists — something didn't clean up")
except Exception:
    print("confirmed gone")
